# 08 — Run the best possession pipeline

This is the organiser-facing executable notebook. It runs the canonical `run_interpolated_flight_experiment.py` pipeline and evaluates the resulting visible-ball low-motion dead-ball candidate.

The pipeline is observation-first: engineered observations → scenario evidence → deterministic anchors → bounded transition hypotheses → final H/A/D/X labels. Kalman estimates missing-ball geometry only; it does not decide possession. Viterbi/HMM/HSMM remain optional decoder diagnostics, not the primary truth-maker.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUNNER = ROOT / 'possession_lab' / 'run_interpolated_flight_experiment.py'
RUNNER_MODULE = 'possession_lab.run_interpolated_flight_experiment'
EVALUATOR = ROOT / 'evaluation' / 'run_evaluation.py'
RESULTS = ROOT / 'evaluation' / 'results'
REPORTS = ROOT / 'evaluation' / 'reports'
BEST_NAME = 'Visible-ball low-motion dead-ball candidate'
BEST_OUTPUT = RESULTS / 'diagnostic_visible_ball_low_motion_dead_ball.csv'
configured_data_root = os.environ.get('MTAH_DATA_DIR')
data_candidates = [Path(configured_data_root)] if configured_data_root else []
data_candidates += [ROOT / 'data', ROOT.parent / 'barca-hack-repo-staging' / 'data']
DATA_ROOT = next((path for path in data_candidates if (path / 'tactical' / 'BAR-ATH_tactical_atd.txt').exists()), None)
assert RUNNER.exists(), RUNNER
assert EVALUATOR.exists(), EVALUATOR
assert DATA_ROOT is not None, f'tactical data not found in: {data_candidates}'
print('project root:', ROOT)
print('runner:', RUNNER)
print('data root:', DATA_ROOT)
print('candidate:', BEST_NAME)

## Execute the candidate pipeline

This cell regenerates the frame-aligned candidate and its diagnostic evidence files. No ground-truth labels are read by the runner.

In [ ]:
runner_environment = os.environ.copy()
runner_environment['MTAH_DATA_DIR'] = str(DATA_ROOT)
pipeline = subprocess.run(
    [sys.executable, '-m', RUNNER_MODULE],
    cwd=ROOT,
    env=runner_environment,
    capture_output=True,
    text=True,
    check=True,
)
print('pipeline completed')
assert BEST_OUTPUT.exists(), BEST_OUTPUT
print('candidate written:', BEST_OUTPUT)

## Public five-minute evaluation

The public labels are used after prediction for evaluation only. They cover frames 11185–18684; all other frames are ungraded `X`.

In [ ]:
evaluation = subprocess.run(
    [sys.executable, str(EVALUATOR), '--gt', 'public'],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print('public evaluation completed')
scoreboard = pd.read_csv(ROOT / 'evaluation' / 'scoreboard.csv')
name_column = 'file' if 'file' in scoreboard.columns else 'submission'
candidate_rows = scoreboard[scoreboard[name_column].astype(str).str.contains('diagnostic_visible_ball_low_motion_dead_ball')]
display(candidate_rows)
assert not candidate_rows.empty, 'candidate is missing from the public scoreboard'

## Transition and mislabel audit

This is an evaluation-only audit. It joins the public labels after the candidate exists, then reports interval errors and the scenario/transition evidence present at those frames. It does not feed labels back into the pipeline.

In [ ]:
from possession_lab.mislabel_audit import build_mislabel_audit

audit = build_mislabel_audit(
    ROOT / 'evaluation' / 'gt' / 'BAR-ATH_possession_gt_public.csv',
    BEST_OUTPUT,
    REPORTS,
    scenario_path=REPORTS / 'scenario_confidence_scores.csv',
)
display(audit['summary'])
display(audit['runs'].sort_values('duration_frames', ascending=False).head(15))
display(audit['context_summary'].head(25))

## Outputs to present

- Candidate submission: `evaluation/results/diagnostic_visible_ball_low_motion_dead_ball.csv`
- Public scoreboard: `evaluation/scoreboard.csv`
- Transition ledger: `evaluation/reports/diagnostic_visible_ball_low_motion_dead_ball_transitions.csv`
- Scenario/trust evidence: `evaluation/reports/scenario_confidence_scores.csv`
- Mislabel intervals: `evaluation/reports/visible_ball_mislabel_runs.csv`

The first-half hand-labelled reference remains a separate diagnostic experiment in `10_first_half_results.ipynb`; it is not used as the presentation benchmark.